# Attention Mechanisms — Foundations
> Section 01 — Topic 03 — foundations

**Prerequisites:** 01-tokenization, 02-embeddings

**What you'll learn:**
- How attention solves the sequence-to-sequence bottleneck
- Scaled dot-product attention from first principles
- Multi-head attention and why heads specialize
- Positional encoding and masking strategies

**What you'll build:**
- Complete multi-head self-attention from scratch in PyTorch

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Attention Intuition — From seq2seq Bottleneck to Attending to Everything

Before attention was introduced, sequence-to-sequence models (like those used for machine translation) relied on an encoder-decoder architecture where the encoder compressed an entire input sequence into a single fixed-length context vector. This vector was the only information the decoder had access to when generating the output sequence. For short sentences this worked reasonably well, but as input sequences grew longer, cramming everything into one vector became a severe bottleneck — important information from early tokens was lost or diluted.

Bahdanau et al. (2014) proposed a solution: instead of forcing the decoder to rely on a single context vector, let it **attend** to all encoder hidden states at each decoding step. The decoder learns to assign different weights (attention scores) to different input positions depending on what it's currently trying to generate. This was the birth of the attention mechanism.

The key insight is elegantly simple: at each step, compute a relevance score between the decoder's current state and every encoder hidden state, normalize these scores into a probability distribution, and take a weighted sum. The result is a context vector that's dynamically tailored to each decoding step.

Vaswani et al. (2017) took this further with the Transformer, eliminating recurrence entirely and using **self-attention** — where a sequence attends to itself — as the core mechanism. This enabled massive parallelization and became the foundation of every modern language model.

In [ ]:
# Simple additive (Bahdanau) attention to build intuition
# Given encoder hidden states and a decoder state, compute attention

class AdditiveAttention(nn.Module):
    """Bahdanau-style additive attention."""
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.W_query = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_key = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)
    
    def forward(self, query, keys):
        # query: (batch, hidden_dim) — decoder state
        # keys: (batch, seq_len, hidden_dim) — encoder states
        query = self.W_query(query).unsqueeze(1)  # (batch, 1, hidden)
        keys_proj = self.W_key(keys)               # (batch, seq_len, hidden)
        scores = self.v(torch.tanh(query + keys_proj)).squeeze(-1)  # (batch, seq_len)
        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights.unsqueeze(1), keys).squeeze(1)
        return context, weights

# Demo
batch, seq_len, hidden = 1, 6, 32
encoder_states = torch.randn(batch, seq_len, hidden)
decoder_state = torch.randn(batch, hidden)

attn = AdditiveAttention(hidden)
context, weights = attn(decoder_state, encoder_states)
print(f"Attention weights: {weights.detach().numpy().round(3)}")
print(f"Context vector shape: {context.shape}")

In [ ]:
# Visualize attention weights
tokens = ["The", "cat", "sat", "on", "the", "mat"]

fig, ax = plt.subplots(figsize=(8, 2))
ax.bar(tokens, weights.detach().numpy()[0])
ax.set_ylabel("Attention Weight")
ax.set_title("Additive Attention: Which encoder tokens does the decoder attend to?")
plt.tight_layout()
plt.show()

## 2. Scaled Dot-Product Attention — Step by Step from Scratch

The Transformer replaced additive attention with **scaled dot-product attention**, which is both simpler and more efficient. The core idea uses three projections of the input: Queries (Q), Keys (K), and Values (V).

Think of it like a retrieval system: you have a **query** (what you're looking for), a set of **keys** (descriptions of available items), and **values** (the actual items). The attention mechanism computes how well each query matches each key, then returns a weighted combination of the values.

The formula is: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$

The scaling factor $\sqrt{d_k}$ is crucial. Without it, when the dimension $d_k$ is large, the dot products grow large in magnitude, pushing the softmax into regions where it has extremely small gradients. The scaling keeps the dot products in a reasonable range for stable training.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Compute scaled dot-product attention.
    
    Args:
        Q: Queries (batch, seq_len_q, d_k)
        K: Keys (batch, seq_len_k, d_k)
        V: Values (batch, seq_len_k, d_v)
        mask: Optional mask (batch, seq_len_q, seq_len_k)
    
    Returns:
        output: Weighted values (batch, seq_len_q, d_v)
        attention_weights: (batch, seq_len_q, seq_len_k)
    """
    d_k = Q.shape[-1]
    
    # Step 1: Compute dot products between queries and keys
    scores = torch.matmul(Q, K.transpose(-2, -1))  # (batch, seq_q, seq_k)
    
    # Step 2: Scale by sqrt(d_k)
    scores = scores / math.sqrt(d_k)
    
    # Step 3: Apply mask (if provided)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Step 4: Softmax to get attention weights
    attention_weights = F.softmax(scores, dim=-1)
    
    # Step 5: Weighted sum of values
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Demo: 4 tokens, dimension 8
seq_len, d_k = 4, 8
Q = torch.randn(1, seq_len, d_k)
K = torch.randn(1, seq_len, d_k)
V = torch.randn(1, seq_len, d_k)

output, weights = scaled_dot_product_attention(Q, K, V)
print(f"Output shape: {output.shape}")
print(f"Attention weights:\n{weights[0].detach().numpy().round(3)}")

In [ ]:
# Visualize: why scaling matters
dims = [8, 64, 512, 2048]
fig, axes = plt.subplots(1, len(dims), figsize=(16, 3))

for ax, d in zip(axes, dims):
    q = torch.randn(1, 10, d)
    k = torch.randn(1, 10, d)
    
    # Unscaled
    raw_scores = torch.matmul(q, k.transpose(-2, -1))
    # Scaled
    scaled_scores = raw_scores / math.sqrt(d)
    
    unscaled_attn = F.softmax(raw_scores, dim=-1)
    scaled_attn = F.softmax(scaled_scores, dim=-1)
    
    ax.hist(unscaled_attn.flatten().detach().numpy(), bins=30, alpha=0.5, label="Unscaled", density=True)
    ax.hist(scaled_attn.flatten().detach().numpy(), bins=30, alpha=0.5, label="Scaled", density=True)
    ax.set_title(f"d_k = {d}")
    ax.legend(fontsize=8)

fig.suptitle("Attention weight distributions: Unscaled vs Scaled", y=1.02)
plt.tight_layout()
plt.show()

## 3. Multi-Head Attention — How Heads Specialize

A single attention function can only capture one type of relationship at a time. **Multi-head attention** runs multiple attention operations in parallel, each with its own learned projection matrices. This allows the model to jointly attend to information from different representation subspaces — one head might focus on syntactic relationships while another captures semantic similarity.

The mechanism splits the model dimension into $h$ heads, each operating on $d_k = d_{\text{model}} / h$ dimensions. The outputs of all heads are concatenated and projected back to the model dimension:

$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$

where $\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$

In practice, research has shown that different attention heads do indeed learn to specialize. Some heads attend primarily to the previous token (positional), some to syntactically related tokens (like subject-verb agreement), and others capture longer-range semantic relationships.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head attention from scratch."""
    
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        # Projection matrices for Q, K, V and output
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.shape[0]
        
        # 1. Linear projections
        Q = self.W_q(query)  # (batch, seq, d_model)
        K = self.W_k(key)
        V = self.W_v(value)
        
        # 2. Split into heads: (batch, seq, d_model) -> (batch, n_heads, seq, d_k)
        Q = Q.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # 3. Scaled dot-product attention per head
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)  # (batch, n_heads, seq_q, seq_k)
        attn_output = torch.matmul(attn_weights, V)  # (batch, n_heads, seq_q, d_k)
        
        # 4. Concatenate heads: (batch, n_heads, seq, d_k) -> (batch, seq, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # 5. Final projection
        output = self.W_o(attn_output)
        
        return output, attn_weights

# Demo
d_model, n_heads, seq_len = 64, 8, 6
mha = MultiHeadAttention(d_model, n_heads)
x = torch.randn(1, seq_len, d_model)

output, attn_weights = mha(x, x, x)  # self-attention
print(f"Input shape:  {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}  (batch, heads, seq_q, seq_k)")

In [ ]:
# Visualize attention patterns across heads
tokens = ["The", "cat", "sat", "on", "a", "mat"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < n_heads:
        im = ax.imshow(attn_weights[0, i].detach().numpy(), cmap='Blues', vmin=0, vmax=1)
        ax.set_xticks(range(len(tokens)))
        ax.set_yticks(range(len(tokens)))
        ax.set_xticklabels(tokens, rotation=45, fontsize=8)
        ax.set_yticklabels(tokens, fontsize=8)
        ax.set_title(f"Head {i}", fontsize=10)

fig.suptitle("Attention Patterns Across Heads (randomly initialized)", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Self-Attention vs Cross-Attention

There are two fundamentally different ways attention is used in Transformers, and understanding the distinction is essential:

**Self-attention** is when a sequence attends to itself. The queries, keys, and values all come from the same input sequence. Every token computes attention scores against every other token in the same sequence. This is how encoder blocks and decoder blocks (for the masked self-attention part) work. Self-attention lets a model capture relationships between tokens within a single sequence — syntactic dependencies, coreference resolution, semantic relationships.

**Cross-attention** is when one sequence attends to a different sequence. The queries come from one sequence (typically the decoder) while the keys and values come from another (typically the encoder output). This is how the decoder in an encoder-decoder model accesses the encoded input. Cross-attention is also used in multimodal models where text attends to image features, or in retrieval-augmented models where the current context attends to retrieved documents.

In [ ]:
# Self-attention: Q, K, V all from the same sequence
d_model, n_heads = 64, 4
self_attn = MultiHeadAttention(d_model, n_heads)

encoder_input = torch.randn(1, 5, d_model)  # "The cat is sleeping"
self_out, self_weights = self_attn(encoder_input, encoder_input, encoder_input)
print(f"Self-attention: input {encoder_input.shape} -> output {self_out.shape}")
print(f"  Each token attends to all {encoder_input.shape[1]} tokens in the same sequence\n")

# Cross-attention: Q from decoder, K/V from encoder
cross_attn = MultiHeadAttention(d_model, n_heads)

decoder_input = torch.randn(1, 3, d_model)   # Decoder generating translation
encoder_output = torch.randn(1, 5, d_model)   # Encoder output from source

cross_out, cross_weights = cross_attn(decoder_input, encoder_output, encoder_output)
print(f"Cross-attention: query {decoder_input.shape}, key/value {encoder_output.shape} -> output {cross_out.shape}")
print(f"  Each decoder token attends to all {encoder_output.shape[1]} encoder tokens")

## 5. Positional Encoding — Sinusoidal and Learned

Attention is inherently **permutation-invariant**: if you shuffle the input tokens, the attention mechanism would produce the same outputs (just shuffled). But word order matters enormously in language — "dog bites man" is very different from "man bites dog." Positional encodings inject information about token position into the representations.

The original Transformer used **sinusoidal positional encodings** — fixed patterns of sine and cosine waves at different frequencies. Each dimension of the positional encoding uses a sinusoid of a different frequency, creating a unique "fingerprint" for each position. The elegant property is that the encoding for position $p + k$ can be expressed as a linear function of the encoding at position $p$, which in theory allows the model to learn relative positions.

The formulas are:
- $PE_{(pos, 2i)} = \sin(pos / 10000^{2i/d_{\text{model}}})$
- $PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i/d_{\text{model}}})$

**Learned positional embeddings** are simply a trainable embedding table, one vector per position. BERT and GPT-2 use learned embeddings. They're simpler and often work just as well, but they fix the maximum sequence length at training time. More modern approaches like RoPE (Rotary Position Embeddings) handle position differently — we'll cover those in the advanced notebook.

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    """Sinusoidal positional encoding from 'Attention Is All You Need'."""
    
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        
        # Compute the div_term: 10000^(2i/d_model)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term)  # Even dimensions
        pe[:, 1::2] = torch.cos(position * div_term)  # Odd dimensions
        
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)
    
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        return x + self.pe[:, :x.size(1)]

# Visualize the sinusoidal patterns
d_model = 64
pe = SinusoidalPositionalEncoding(d_model)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap of positional encodings
im = ax1.imshow(pe.pe[0, :50, :].numpy(), aspect='auto', cmap='RdBu')
ax1.set_xlabel('Dimension')
ax1.set_ylabel('Position')
ax1.set_title('Sinusoidal Positional Encoding')
plt.colorbar(im, ax=ax1)

# Individual dimensions
positions = range(50)
for dim in [0, 1, 4, 5, 20, 21]:
    ax2.plot(positions, pe.pe[0, :50, dim].numpy(), label=f'dim {dim}')
ax2.set_xlabel('Position')
ax2.set_ylabel('Value')
ax2.set_title('Individual Dimensions (different frequencies)')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Learned positional embeddings (much simpler)
class LearnedPositionalEmbedding(nn.Module):
    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        self.embedding = nn.Embedding(max_len, d_model)
    
    def forward(self, x):
        # x: (batch, seq_len, d_model)
        positions = torch.arange(x.size(1), device=x.device)
        return x + self.embedding(positions)

# Compare: similarity between position encodings
sin_pe = SinusoidalPositionalEncoding(d_model)
learned_pe = LearnedPositionalEmbedding(512, d_model)

# Cosine similarity between positions for sinusoidal
sin_vecs = sin_pe.pe[0, :20]  # First 20 positions
sin_sim = F.cosine_similarity(sin_vecs.unsqueeze(0), sin_vecs.unsqueeze(1), dim=-1)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sin_sim.numpy(), cmap='RdBu', vmin=-1, vmax=1)
ax.set_title('Sinusoidal PE: Cosine Similarity Between Positions')
ax.set_xlabel('Position')
ax.set_ylabel('Position')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()
print("Notice: nearby positions have higher similarity, and it decays with distance.")

## 6. Masking — Padding Masks and Causal Masks

Masking is essential for making attention work correctly in two scenarios:

**Padding masks** handle variable-length sequences in a batch. When sequences have different lengths, shorter ones are padded with special tokens. Without masking, the model would attend to these meaningless pad tokens. A padding mask sets attention scores to $-\infty$ for padded positions, which softmax converts to zero weight — effectively ignoring those positions.

**Causal masks** (also called "look-ahead masks") prevent the decoder from seeing future tokens during autoregressive generation. When training a language model, we process the entire sequence at once for efficiency, but the prediction at position $t$ should only depend on positions $0$ through $t-1$. The causal mask is an upper-triangular matrix of $-\infty$ values that blocks all future positions. This is why decoder-only models like GPT work: during training they see the whole sequence but each position can only attend to earlier positions, simulating autoregressive generation.

In [ ]:
# Padding mask: ignore pad tokens in attention
def create_padding_mask(seq_lens, max_len):
    """Create padding mask from sequence lengths.
    Returns mask of shape (batch, 1, 1, max_len) for broadcasting.
    """
    batch_size = len(seq_lens)
    mask = torch.zeros(batch_size, max_len)
    for i, length in enumerate(seq_lens):
        mask[i, :length] = 1
    return mask.unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, max_len)

# Causal mask: prevent attending to future tokens
def create_causal_mask(seq_len):
    """Create lower-triangular causal mask.
    Returns mask of shape (1, 1, seq_len, seq_len) for broadcasting.
    """
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)

# Visualize both mask types
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Padding mask for batch of 2 sequences (lengths 4 and 6 out of max 6)
pad_mask = create_padding_mask([4, 6], 6)
axes[0].imshow(pad_mask[0, 0, 0].unsqueeze(0).expand(6, -1).numpy(), cmap='Greens', vmin=0, vmax=1)
axes[0].set_title('Padding Mask (seq_len=4, max=6)')
axes[0].set_xlabel('Key Position')
axes[0].set_ylabel('Query Position')

# Causal mask
causal = create_causal_mask(6)
axes[1].imshow(causal[0, 0].numpy(), cmap='Greens', vmin=0, vmax=1)
axes[1].set_title('Causal Mask (6 tokens)')
axes[1].set_xlabel('Key Position')
axes[1].set_ylabel('Query Position')

# Combined mask (causal + padding)
combined = causal * pad_mask[0:1]
axes[2].imshow(combined[0, 0].numpy(), cmap='Greens', vmin=0, vmax=1)
axes[2].set_title('Combined (Causal + Padding)')
axes[2].set_xlabel('Key Position')
axes[2].set_ylabel('Query Position')

for ax in axes:
    ax.set_xticks(range(6))
    ax.set_yticks(range(6))

plt.tight_layout()
plt.show()
print("Green = can attend, White = blocked")

In [ ]:
# Demonstrate masking effect on attention weights
seq_len, d_k = 6, 32
Q = torch.randn(1, seq_len, d_k)
K = torch.randn(1, seq_len, d_k)
V = torch.randn(1, seq_len, d_k)

# Without mask — every token attends to every other token
_, weights_no_mask = scaled_dot_product_attention(Q, K, V)

# With causal mask — each token only attends to itself and earlier tokens
causal_mask = create_causal_mask(seq_len)
_, weights_causal = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
tokens = ["The", "cat", "sat", "on", "a", "mat"]

for ax, weights, title in zip(axes, 
    [weights_no_mask, weights_causal], 
    ["No Mask (bidirectional)", "Causal Mask (autoregressive)"]):
    im = ax.imshow(weights[0].detach().numpy(), cmap='Blues', vmin=0)
    ax.set_xticks(range(6))
    ax.set_yticks(range(6))
    ax.set_xticklabels(tokens, rotation=45)
    ax.set_yticklabels(tokens)
    ax.set_title(title)
    ax.set_xlabel('Attends to')
    ax.set_ylabel('Token')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

## Putting It All Together — Complete Self-Attention Block

Let's combine everything into a complete Transformer-style self-attention block. This is the fundamental building block that gets stacked to create modern language models. A complete block includes multi-head self-attention, residual connections, and layer normalization.

In [ ]:
class SelfAttentionBlock(nn.Module):
    """Complete self-attention block with residual connection and layer norm."""
    
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Pre-norm style (modern convention)
        normed = self.norm(x)
        attn_out, attn_weights = self.attention(normed, normed, normed, mask)
        output = x + self.dropout(attn_out)  # Residual connection
        return output, attn_weights

# Stack multiple blocks
d_model, n_heads, n_layers = 128, 8, 4
blocks = nn.ModuleList([SelfAttentionBlock(d_model, n_heads) for _ in range(n_layers)])

x = torch.randn(2, 10, d_model)  # batch=2, seq_len=10
causal_mask = create_causal_mask(10)

all_weights = []
for block in blocks:
    x, weights = block(x, causal_mask)
    all_weights.append(weights)

print(f"Input shape:  {torch.randn(2, 10, d_model).shape}")
print(f"Output shape: {x.shape}")
print(f"Collected attention weights from {len(all_weights)} layers")
print(f"Each weight tensor: {all_weights[0].shape}  (batch, heads, seq_q, seq_k)")

In [ ]:
# Visualize attention across layers (head 0)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, (ax, w) in enumerate(zip(axes, all_weights)):
    ax.imshow(w[0, 0].detach().numpy(), cmap='Blues', vmin=0)
    ax.set_title(f'Layer {i}, Head 0')
    ax.set_xlabel('Key')
    if i == 0:
        ax.set_ylabel('Query')

fig.suptitle('Attention patterns across layers (randomly initialized)', y=1.02)
plt.tight_layout()
plt.show()
print("After training, different layers capture different types of relationships.")

## Summary

**Key takeaways:**
- Attention replaces the fixed-size bottleneck with dynamic, content-based weighting
- Scaled dot-product attention uses Q, K, V projections with $\sqrt{d_k}$ scaling for stable gradients
- Multi-head attention runs parallel attention in different subspaces — heads specialize
- Self-attention (same sequence) enables encoder/decoder blocks; cross-attention (different sequences) connects encoder to decoder
- Positional encodings inject position information since attention is permutation-invariant
- Causal masking enables autoregressive training by preventing future token access

**Next:** [03-attention-mechanisms/advanced.ipynb](advanced.ipynb) — MQA, GQA, Flash Attention, and other efficient attention variants used in production models